# Classifying news articles with Naive Bayes

Once text data has been converted into numerical features using the natural language processing techniques discussed in the previous sections, text classification works just like any other classification task.

## Imports

In [1]:
%matplotlib inline

from pathlib import Path

import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

## News article classification

We start with an illustration of the Naive Bayes model for news article classification using the BBC articles that we read as before to obtain a DataFrame with 2,225 articles from 5 categories.

### Read BBC articles

In [2]:
DATA_DIR = Path("..", "data")

In [3]:
path = DATA_DIR / "bbc"
files = sorted(list(path.glob("**/*.txt")))
doc_list = []
for i, file in enumerate(files):
    topic = file.parts[-2]
    article = file.read_text(encoding="latin1").split("\n")
    heading = article[0].strip()
    body = " ".join([l.strip() for l in article[1:]])
    doc_list.append([topic, heading, body])

In [4]:
docs = pd.DataFrame(doc_list, columns=["topic", "heading", "body"])
docs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   topic    2225 non-null   object
 1   heading  2225 non-null   object
 2   body     2225 non-null   object
dtypes: object(3)
memory usage: 52.3+ KB


### Create stratified train-test split

We split the data into the default 75:25 train-test sets, ensuring that the test set classes closely mirror the train set:

In [5]:
y = pd.factorize(docs.topic)[0]
X = docs.body
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, stratify=y)

### Vectorize text data

We proceed to learn the vocabulary from the training set and transforming both dataset using the CountVectorizer with default settings to obtain almost 26,000 features:

In [6]:
vectorizer = CountVectorizer()
X_train_dtm = vectorizer.fit_transform(X_train)
X_test_dtm = vectorizer.transform(X_test)

In [7]:
X_train_dtm.shape, X_test_dtm.shape

((1668, 25951), (557, 25951))

### Train Multi-class Naive Bayes model

In [99]:
nb = MultinomialNB(alpha=0.5, fit_prior=True)
nb.fit(X_train_dtm, y_train)
y_pred_class = nb.predict(X_test_dtm)

### Evaluate Results

We evaluate the multiclass predictions using accuracy to find the default classifier achieved almost 98%:

#### Accuracy

In [100]:
accuracy_score(y_test, y_pred_class)

0.9730700179533214

#### Confusion matrix

In [101]:
pd.DataFrame(confusion_matrix(y_true=y_test, y_pred=y_pred_class))

,0,1,2,3,4
0,121,0,5,0,2
1,0,94,2,0,1
2,1,0,103,0,0
3,0,0,1,127,0
4,0,1,2,0,97


In [13]:
from naive_bayes.lib.naive_bayes import TWCNB
from naive_bayes.lib.const import *

In [24]:
from collections import defaultdict
import numpy as np


def format_training_data(X_train: pd.Series, y_train: np.ndarray) -> dict:
    """
    Transforms training data into the format required by the naive_bayes library.

    Args:
        X_train: A pandas Series where each item is a sentence (document).
        y_train: A pandas Series with the corresponding class labels.

    Returns:
        A dictionary where keys are class labels and values are lists of
        tokenized documents.
        e.g., {class1: [[word1, word2], [word3, word4]], class2: ...}
    """
    # A simple space-based tokenizer. You can replace this with a more
    # sophisticated tokenizer if needed (e.g., from NLTK or spaCy).
    tokenized_docs = X_train.str.split()

    # Use defaultdict to automatically handle new classes
    grouped_data = defaultdict(list)

    # Iterate over the labels and tokenized docs to build the dictionary
    for label, doc_tokens in zip(y_train, tokenized_docs):
        grouped_data[label].append(doc_tokens)

    return dict(grouped_data)


# --- Example Usage ---
train_data = format_training_data(X_train, y_train)

In [28]:
def format_test_data(X_test: pd.Series) -> list:
    """
    Transforms test data into the format required by the naive_bayes library.

    Args:
        X_test: A pandas Series where each item is a sentence (document).

    Returns:
        A list of lists, where each inner list contains the tokens of a document.
        e.g., [[word1, word2], [word3, word4], ...]
    """
    # A simple space-based tokenizer
    tokenized_docs = X_test.str.split()
    return tokenized_docs.tolist()


# --- Example Usage ---
test_data = format_test_data(X_test)

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [145]:
def advanced_tokenize(
    text,
    lowercase=True,
    remove_punctuation=True,
    remove_numbers=True,
    lemmatize=True,
    remove_stopwords=True,
):
    """
    Advanced tokenization with multiple preprocessing options.

    Args:
        text (str): Input text
        lowercase (bool): Convert to lowercase
        remove_punctuation (bool): Remove punctuation
        remove_numbers (bool): Remove numbers
        lemmatize (bool): Apply lemmatization
        remove_stopwords (bool): Remove stopwords

    Returns:
        list: List of processed tokens
    """
    if pd.isna(text):
        return []

    # Convert to string if needed
    text = str(text)

    # Lowercase
    if lowercase:
        text = text.lower()

    # Remove punctuation
    if remove_punctuation:
        text = re.sub(r"[^\w\s]", " ", text)

    # Remove numbers
    if remove_numbers:
        text = re.sub(r"\d+", "", text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords
    if remove_stopwords:
        stop_words = set(stopwords.words("english"))
        tokens = [token for token in tokens if token not in stop_words]

    # Lemmatization
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(token) for token in tokens]

    # Remove empty tokens and short tokens
    tokens = [token for token in tokens if len(token) > 1]

    return tokens


def format_training_data_advanced(
    X_train,
    y_train,
    lowercase=True,
    remove_punctuation=True,
    remove_numbers=True,
    lemmatize=True,
    remove_stopwords=True,
):
    """
    Transforms training data with advanced preprocessing.

    Args:
        X_train: A pandas Series where each item is a sentence (document).
        y_train: A pandas Series with the corresponding class labels.
        lowercase, remove_punctuation, remove_numbers, lemmatize, remove_stopwords:
            Preprocessing options passed to advanced_tokenize

    Returns:
        A dictionary where keys are class labels and values are lists of
        tokenized documents.
    """
    # Apply advanced tokenization
    tokenized_docs = X_train.apply(
        lambda x: advanced_tokenize(
            x,
            lowercase,
            remove_punctuation,
            remove_numbers,
            lemmatize,
            remove_stopwords,
        )
    )

    # Group by class
    grouped_data = defaultdict(list)
    for label, doc_tokens in zip(y_train, tokenized_docs):
        grouped_data[label].append(doc_tokens)

    return dict(grouped_data)


def format_test_data_advanced(
    X_test,
    lowercase=True,
    remove_punctuation=True,
    remove_numbers=True,
    lemmatize=True,
    remove_stopwords=True,
):
    """
    Transforms test data with advanced preprocessing.

    Args:
        X_test: A pandas Series where each item is a sentence (document).
        lowercase, remove_punctuation, remove_numbers, lemmatize, remove_stopwords:
            Preprocessing options passed to advanced_tokenize

    Returns:
        A list of lists, where each inner list contains the processed tokens.
    """
    tokenized_docs = X_test.apply(
        lambda x: advanced_tokenize(
            x,
            lowercase,
            remove_punctuation,
            remove_numbers,
            lemmatize,
            remove_stopwords,
        )
    )
    return tokenized_docs.tolist()


# Example usage with recommended settings for financial text
def preprocess_financial_text(X_train, y_train, X_test):
    """
    Preprocessing specifically tuned for financial text data.
    """
    # For financial text, we might want to keep numbers and some punctuation
    # but still clean up the text
    train_data = format_training_data_advanced(
        X_train,
        y_train,
        lowercase=True,
        remove_punctuation=True,  # Keep some punctuation for financial terms
        remove_numbers=True,  # Keep numbers for prices, dates, etc.
        lemmatize=True,
        remove_stopwords=True,
    )

    test_data = format_test_data_advanced(
        X_test,
        lowercase=True,
        remove_punctuation=True,
        remove_numbers=True,
        lemmatize=True,
        remove_stopwords=True,
    )

    return train_data, test_data

In [146]:
train_data, test_data = preprocess_financial_text(X_train, y_train, X_test)

In [178]:
params = NBParam(LEARNING_MAP, FS_NO, 1000, 30000, 2)

In [179]:
model = TWCNB(param=params, use_cython=True)

In [180]:
model.learn(train=train_data)

100%|██████████| 5/5 [00:00<00:00, 11.20it/s]


In [181]:
y_pred_class2 = model.predict(test=test_data, get_score=False)

100%|██████████| 557/557 [00:00<00:00, 3809.32it/s]


In [182]:
accuracy_score(y_test, y_pred_class2)

0.9748653500897666

In [183]:
pd.DataFrame(confusion_matrix(y_true=y_test, y_pred=y_pred_class2))

,0,1,2,3,4
0,122,0,5,0,1
1,0,93,2,0,2
2,1,0,103,0,0
3,0,0,0,128,0
4,0,1,2,0,97


In [144]:
pd.DataFrame(confusion_matrix(y_test, y_pred_class))

,0,1,2,3,4
0,121,0,5,0,2
1,0,94,2,0,1
2,1,0,103,0,0
3,0,0,1,127,0
4,0,1,2,0,97
